In [28]:
import polars as pl
import pandas as pd
import numpy as np
import os, glob
import re
import plotly.express as px
import polars as pl

In [179]:
# directories
cwd = os.getcwd() # directory this file is saved in
data_dir = os.path.join(cwd, "BCDC-Metadata") # BCDC data folder
metadata_dir = os.path.join(data_dir, 'Sample-Inventory') # directory with metadata

df = pd.read_csv(os.path.join(metadata_dir, 'BCDC_Metadata_2022Q3.csv'), encoding='unicode_escape')

### use a data cleaning function here that takes both this dataset and CV
# 1. get rid of all the special characters in CV columns and column names
# 2. cv and data columns both to lower case
# 3. way to identify extremely similar words (i.e. brain vs brains)
### in simple code logic...
# list of cols with controlled values
controlled_cols = [re.sub("\([^)]*\)", "", x).strip() for x in list(filter(lambda x : 'CV' in x, df.columns))]
df.columns = df.columns.str.replace(r"\([^)]*\)","").str.strip() # get rid of (CV) in data columns
df['Sample Type'] = df['Sample Type'].str.replace("[^A-Za-z0-9 ]+", " ").str.lower() # special characters into spaces
# to-do: do the same for all CV columns
df['Subspecimen Type'] = df['Subspecimen Type'].str.replace("[^A-Za-z0-9 ]+", " ").str.lower() # special characters into spaces
df['Total Processed Subspecimens'] = df['Total Processed Subspecimens'].astype(str).str.replace(',', '').astype(float) # cell counts is numeric

df['category'] = ''
df

/var/folders/76/jdv5bsqj4_9dk0rm_g_vp9gm0000gp/T/ipykernel_20593/2455064548.py:6: DtypeWarning: Columns (4,5,6,7,8,9,12,14,20,21,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(metadata_dir, 'BCDC_Metadata_2022Q3.csv'), encoding='unicode_escape')
/var/folders/76/jdv5bsqj4_9dk0rm_g_vp9gm0000gp/T/ipykernel_20593/2455064548.py:15: FutureWarning: The default value of regex will change from True to False in a future version.
  df.columns = df.columns.str.replace(r"\([^)]*\)","").str.strip() # get rid of (CV) in data columns
/var/folders/76/jdv5bsqj4_9dk0rm_g_vp9gm0000gp/T/ipykernel_20593/2455064548.py:16: FutureWarning: The default value of regex will change from True to False in a future version.
  df['Sample Type'] = df['Sample Type'].str.replace("[^A-Za-z0-9 ]+", " ").str.lower() # special characters into spaces
/var/folders/76/jdv5bsqj4_9dk0rm_g_vp9gm0000gp/T/ipykernel_20593/2455064548.py:18: FutureWarning: The default value 

,Sample ID,Sample Type,Species,Species NCBI Taxonomy ID,Parent Specimen ID,Parent Specimen Type,Subject ID,Age,Sex,Genotype,...,Organization,Investigator,Grant Number,Data Collection,R24 Name,R24 link,Comments,Metadata Submission,Unnamed: 23,category
0,SW170829-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170829-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,
1,SW170829-02A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170829-02A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,
2,SW170910-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170910-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,
3,SW170917-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170917-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,
4,SW171101-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW171101-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155658,1138845290,cell in slice,human,NCBI:txid9606,NaN,NaN,H21.28.030,41,Male,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-10,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1138845290,NaN,2022Q3,NaN,
155659,1126880187,cell in slice,human,NCBI:txid9606,NaN,NaN,H21.28.031,72,Male,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-11,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1126880187,NaN,2022Q3,NaN,
155660,1126880246,cell in slice,human,NCBI:txid9606,NaN,NaN,H21.29.201,45,Male,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-12,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1126880246,NaN,2022Q3,NaN,
155661,1159995431,cell in slice,human,NCBI:txid9606,NaN,NaN,H22.03.301,20,Female,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-13,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1159995431,NaN,2022Q3,NaN,


In [180]:
### Mouse brain atlas ontology file to get anatomical structures
### from http://lims2/structure_graphs
mouse_ontology = pd.read_csv(os.path.join(cwd, 'Mouse.csv'), sep='delimiter', engine='python')
cols = list(mouse_ontology.columns.str.split(',').tolist()[0])
mouse_ontology = mouse_ontology.iloc[:,0].str.split(',', expand=True)
mouse_ontology.iloc[:,9] = mouse_ontology.iloc[:,9:].replace('', np.nan).apply(lambda x: pd.Series(x.dropna().values), 1).iloc[:,0]
mouse_ontology = mouse_ontology.iloc[:, 0:10]
mouse_ontology.columns = cols
mouse_ontology

,InDel,ID,Acronym,Hemisphere,red,green,blue,st_level,st_order,Structures
0,,997,root,,255,255,255,0,,root
1,,8,grey,,191,218,227,1,,Basic cell groups and regions
2,,567,CH,,176,240,255,2,,Cerebrum
3,,688,CTX,,176,255,184,3,,Cerebral cortex
4,,695,CTXpl,,112,255,112,4,,Cortical plate
...,...,...,...,...,...,...,...,...,...,...
1322,,49,ipf,,170,170,170,8,,intraparafloccular fissure
1323,,57,pms,,170,170,170,8,,paramedian sulcus
1324,,65,pfs,,170,170,170,8,,parafloccular sulcus
1325,,624,IPF,,170,170,170,7,,Interpeduncular fossa


In [181]:
### containers
# brain and cell for now...add more later
brain_df = pd.DataFrame(columns = df.columns)
cell_df = pd.DataFrame(columns = df.columns)

In [183]:
df.loc[df['Subspecimen Type'].isin(['brain', 'brains', 'brain hemisphere', 'whole brain', 'spinal cord']), 'category']='Brain'
df.loc[(df['Sample Type'].str.contains("cell", na = False)) | (df['Subspecimen Type'].str.contains("cell", na = False)), 'category']='Cell'
df

,Sample ID,Sample Type,Species,Species NCBI Taxonomy ID,Parent Specimen ID,Parent Specimen Type,Subject ID,Age,Sex,Genotype,...,Organization,Investigator,Grant Number,Data Collection,R24 Name,R24 link,Comments,Metadata Submission,Unnamed: 23,category
0,SW170829-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170829-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,Brain
1,SW170829-02A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170829-02A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,Brain
2,SW170910-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170910-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,Brain
3,SW170917-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW170917-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,Brain
4,SW171101-01A,whole brain,mouse,NCBI:txid10090,NaN,NaN,SW171101-01A,NaN,NaN,wt/wt,...,University of Southern California,Hong-Wei Dong,1U19MH114821-01,huang_antero,BIL,http://download.brainimagelibrary.org/biccn/hu...,NaN,2018Q3,NaN,Brain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155658,1138845290,cell in slice,human,NCBI:txid9606,NaN,NaN,H21.28.030,41,Male,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-10,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1138845290,NaN,2022Q3,NaN,Cell
155659,1126880187,cell in slice,human,NCBI:txid9606,NaN,NaN,H21.28.031,72,Male,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-11,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1126880187,NaN,2022Q3,NaN,Cell
155660,1126880246,cell in slice,human,NCBI:txid9606,NaN,NaN,H21.29.201,45,Male,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-12,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1126880246,NaN,2022Q3,NaN,Cell
155661,1159995431,cell in slice,human,NCBI:txid9606,NaN,NaN,H22.03.301,20,Female,NaN,...,Allen Institute for Brain Science,Ed Lein,1U01MH114812-13,lein_human_pseq_m,BIL,/bil/lz/rdaibs/efb9b12ba2fab63d/1159995431,NaN,2022Q3,NaN,Cell


In [184]:
## for the rows not labelled as brains or cells, set conditions for further investigations
conditions_df = df[df['category'] == ''].groupby(['Sample Type', 'Subspecimen Type']).size().reset_index().rename(columns={0:'count'})
conditions_df

,Sample Type,Subspecimen Type,count
0,brain hemisphere,brain section set,3
1,brain region,brain region,2
2,brain region,brain slice,17
3,brain region,nuclei,134
4,brain region,tissue sample,3
5,brain section set,brain section set,35
6,brain slice,brain slice,148
7,library,library,147
8,library,nuclei,49
9,reconstruction,reconstruction,2


In [185]:
conditions_dict = conditions_df.groupby('Sample Type')['Subspecimen Type'].apply(list).to_dict()    
conditions_dict 

{'brain hemisphere': ['brain section set'],
 'brain region': ['brain region', 'brain slice', 'nuclei', 'tissue sample'],
 'brain section set': ['brain section set'],
 'brain slice': ['brain slice'],
 'library': ['library', 'nuclei'],
 'reconstruction': ['reconstruction', 'reconstructions'],
 'reconstructions': ['reconstructions'],
 'specimen': ['brain slice',
  'bulk nucleus',
  'reconstruction',
  'specimen',
  'spinal cord slice',
  'tissue sample'],
 'spinal cord': ['spinal cord slice'],
 'spinal cord slice': ['neuron']}

In [186]:
## just a test
conditions_dict = conditions_df.groupby('Sample Type')['Subspecimen Type'].apply(list).to_dict()    
for key, value in conditions_dict.items():
    sample_df = df[df['Sample Type'] == key]
    print(sample_df[sample_df['Subspecimen Type'].isin(['brain', 'brains', 'brain hemisphere', 'whole brain', 'spinal cord'])])

            Sample ID       Sample Type Species Species NCBI Taxonomy ID  \
69100      I46_LH_MRI  brain hemisphere   human            NCBI:txid9606   
69101      I48_LH_MRI  brain hemisphere   human            NCBI:txid9606   
69102      I56_LH_MRI  brain hemisphere   human            NCBI:txid9606   
69103      I55_LH_MRI  brain hemisphere   human            NCBI:txid9606   
69104      I52_RH_MRI  brain hemisphere   human            NCBI:txid9606   
69105      I53_LH_MRI  brain hemisphere   human            NCBI:txid9606   
69106      I41_LH_MRI  brain hemisphere   human            NCBI:txid9606   
69107   EXC022_LH_MRI  brain hemisphere   human            NCBI:txid9606   
70180       KC001_MRI  brain hemisphere   human            NCBI:txid9606   
70339      I38_LH_MRI  brain hemisphere   human            NCBI:txid9606   
70340      I59_RH_MRI  brain hemisphere   human            NCBI:txid9606   
70341      I60_RH_MRI  brain hemisphere   human            NCBI:txid9606   
150361     I

In [31]:
brain_anat = mouse_ontology[mouse_ontology['Structures'].str.contains('brain', case = False)]['Acronym'].unique()
df[df['Anatomical Structure'].isin([brain_anat])]['Subspecimen Type'].unique() # returns nothing?
# and where does anatomical structure = 'Br' come from?
df[df['Anatomical Structure'].isin(['Br'])]['Subspecimen Type'].unique()

array(['brain hemisphere'], dtype=object)

In [187]:
### next - if column N says 'section', 'region', 'slice', 'sample', 'specimen'
conditions_dict

{'brain hemisphere': ['brain section set'],
 'brain region': ['brain region', 'brain slice', 'nuclei', 'tissue sample'],
 'brain section set': ['brain section set'],
 'brain slice': ['brain slice'],
 'library': ['library', 'nuclei'],
 'reconstruction': ['reconstruction', 'reconstructions'],
 'reconstructions': ['reconstructions'],
 'specimen': ['brain slice',
  'bulk nucleus',
  'reconstruction',
  'specimen',
  'spinal cord slice',
  'tissue sample'],
 'spinal cord': ['spinal cord slice'],
 'spinal cord slice': ['neuron']}

In [200]:
## dictionary of what technique processes what subspecimen type
conditions_df = df.groupby(['Technique', 'Subspecimen Type']).size().reset_index().rename(columns={0:'count'}).drop_duplicates()
tech_dict = conditions_df.groupby('Technique')['Subspecimen Type'].apply(list).to_dict()
tech_dict

{'10X Genomics Multiome': ['cell nucleus'],
 '10X Genomics Multiome; ATAC-seq': ['cell nucleus'],
 '10X Genomics Multiome; RNAseq': ['cell nucleus'],
 "10x Chromium 3' v2 sequencing": ['cell',
  'cell body',
  'cell nucleus',
  'whole cell'],
 "10x Chromium 3' v3 sequencing": ['cell body',
  'cell nucleus',
  'cells',
  'library',
  'nuclei',
  'whole cell'],
 "10x Chromium 3' v3.1 sequencing": ['cells'],
 '10x Chromium Multiome': ['cell nucleus'],
 'ATAC-seq': ['cell nucleus', 'nuclei', 'whole cell'],
 'DNAseq': ['tissue sample'],
 'Drop-seq': ['cell nucleus'],
 'FISH': ['cells', 'whole cell'],
 'Histology': ['brain slice'],
 'MERFISH': ['brain slice', 'tissue sample', 'whole cell'],
 'MORF genetic sparse labeling': ['brain section set'],
 'MRI': ['brain', 'brain hemisphere', 'whole brain'],
 'OCT': ['brain slice'],
 'OLST': ['brain'],
 'PATCH-Seq': ['cell in slice', 'cell nucleus'],
 'PATCH-seq': ['cell in slice'],
 'Patch-seq;SMART-seq v4': ['cell nucleus'],
 'Patch-seq;neuron morph

In [228]:
brain_tech_list = []
cell_tech_list = []
tech_category_dict = {}

seq_list = []
sample_tech_list = []
nucleus_tech_list = []
recons_tech_list = []
for key, values in tech_dict.items():
    # sequencing techniques
    seq_list.append(key) if 'seq' in key else None
    # all-cell techniques
    cell_match = [x for x in values if any(word in x for word in ['cell', 'nucleus', 'nuclei'])]
    if len(cell_match) == len(values): # if all values in dict says cell in some way; nucleus/nuclei are also cells
        cell_tech_list.append(key)
    # all-brain techniques
    brain_match = [x for x in values if "brain" in x] # if all values in dict say brain in some way
    if len(brain_match) == len(values): 
        brain_tech_list.append(key)
    # tissue/organ techniques
    sample_list = [x for x in values if any(word in x for word in ['brain', 'tissue', 'sample', 'library', 'specimen', 'slice'])]
    if sample_list:
        sample_tech_list.append(key)
    # nucleus tehniques
    nu_list = [x for x in values if any(word in x for word in ['nucleus', 'nuclei'])]
    if nu_list:
        nucleus_tech_list.append(key)
    # reconstructions
    recons_list = [x for x in values if any(word in x for word in ['reconstruction', 'reconstructions'])]
    if recons_list:
        recons_tech_list.append(key)
        

tech_category_dict['sequencing'] = seq_list # dictionary containing sequencing techniques
tech_category_dict['tissue_sample_techniques'] = sample_tech_list # techniques were tissue slices can be used
tech_category_dict['nucleus_techniques'] = nucleus_tech_list # nucleus techniques
tech_category_dict['reconstruction_techs'] = recons_tech_list

In [229]:
tech_category_dict

{'sequencing': ['10X Genomics Multiome; ATAC-seq',
  '10X Genomics Multiome; RNAseq',
  "10x Chromium 3' v2 sequencing",
  "10x Chromium 3' v3 sequencing",
  "10x Chromium 3' v3.1 sequencing",
  'ATAC-seq',
  'DNAseq',
  'Drop-seq',
  'PATCH-seq',
  'Patch-seq;SMART-seq v4',
  'Patch-seq;neuron morphology reconstruction',
  'Patch-seq;whole cell patch clamp',
  'SHARE-seq',
  'SMART-seq v4',
  'SNARE-seq2 (ATAC-seq)',
  'SNARE-seq2 (RNA-seq)',
  'SNARE-seq2;ATAC-seq',
  'SNARE-seq2;RNAseq',
  'Slide-seq',
  'mC-seq10',
  'mC-seq2',
  'mC-seq2;retrograde tracing',
  'mC-seq3',
  'mC-seq4',
  'mC-seq5',
  'mC-seq6',
  'mC-seq7',
  'mC-seq8',
  'mC-seq9',
  'sci-RNA-seq3',
  'seqFISH'],
 'tissue_sample_techniques': ["10x Chromium 3' v3 sequencing",
  'DNAseq',
  'Histology',
  'MERFISH',
  'MORF genetic sparse labeling',
  'MRI',
  'OCT',
  'OLST',
  'PATCH-Seq',
  'PATCH-seq',
  'Patch-seq;neuron morphology reconstruction',
  'Patch-seq;whole cell patch clamp',
  'STPT',
  'STPT;enhancer

In [207]:
tech_dict

{'10X Genomics Multiome': ['cell nucleus'],
 '10X Genomics Multiome; ATAC-seq': ['cell nucleus'],
 '10X Genomics Multiome; RNAseq': ['cell nucleus'],
 "10x Chromium 3' v2 sequencing": ['cell',
  'cell body',
  'cell nucleus',
  'whole cell'],
 "10x Chromium 3' v3 sequencing": ['cell body',
  'cell nucleus',
  'cells',
  'library',
  'nuclei',
  'whole cell'],
 "10x Chromium 3' v3.1 sequencing": ['cells'],
 '10x Chromium Multiome': ['cell nucleus'],
 'ATAC-seq': ['cell nucleus', 'nuclei', 'whole cell'],
 'DNAseq': ['tissue sample'],
 'Drop-seq': ['cell nucleus'],
 'FISH': ['cells', 'whole cell'],
 'Histology': ['brain slice'],
 'MERFISH': ['brain slice', 'tissue sample', 'whole cell'],
 'MORF genetic sparse labeling': ['brain section set'],
 'MRI': ['brain', 'brain hemisphere', 'whole brain'],
 'OCT': ['brain slice'],
 'OLST': ['brain'],
 'PATCH-Seq': ['cell in slice', 'cell nucleus'],
 'PATCH-seq': ['cell in slice'],
 'Patch-seq;SMART-seq v4': ['cell nucleus'],
 'Patch-seq;neuron morph

In [230]:
# dictionary of the ones that are not brain or cell
# likely 'specimens' that need to be further categorized
{key: tech_dict[key] for key in set(tech_dict.keys()) - set(brain_tech_list + cell_tech_list)}

{'DNAseq': ['tissue sample'],
 "10x Chromium 3' v3 sequencing": ['cell body',
  'cell nucleus',
  'cells',
  'library',
  'nuclei',
  'whole cell'],
 'VISor;light sheet microscopy': ['spinal cord'],
 'TRIO tracing': ['brain', 'brains', 'spinal cord', 'whole brain'],
 'whole cell patch clamp': ['neuron'],
 'MERFISH': ['brain slice', 'tissue sample', 'whole cell'],
 'fMOST': ['whole brain', 'whole cell'],
 'seqFISH': ['spinal cord slice'],
 'sci-RNA-seq3': ['specimen'],
 'neuron morphology reconstruction': ['reconstruction',
  'reconstructions',
  'whole cell']}

In [199]:
df[df['Technique'].isin(brain_tech_list)]['Subspecimen Type']

671       brain section set
981             whole brain
982             whole brain
983             whole brain
984             whole brain
                ...        
150486                  NaN
150776    brain section set
150777                  NaN
155645    brain section set
155646    brain section set
Name: Subspecimen Type, Length: 1745, dtype: object

In [177]:
brain_tech_list + cell_tech_list

['Histology',
 'MORF genetic sparse labeling',
 'MRI',
 'OCT',
 'OLST',
 'STPT',
 'STPT;enhancer virus labeling',
 'anterograde tracing',
 'cre-dependent anterograde tracing',
 'histology',
 'light sheet microscopy',
 'mouselight',
 'retrograde tracing',
 'retrograde transsynaptic tracing',
 'smFISH',
 'stereology',
 'triple anterograde',
 '10X Genomics Multiome',
 '10X Genomics Multiome; ATAC-seq',
 '10X Genomics Multiome; RNAseq',
 "10x Chromium 3' v2 sequencing",
 "10x Chromium 3' v3.1 sequencing",
 '10x Chromium Multiome',
 'ATAC-seq',
 'Drop-seq',
 'FISH',
 'PATCH-Seq',
 'PATCH-seq',
 'Patch-seq;SMART-seq v4',
 'Patch-seq;neuron morphology reconstruction',
 'Patch-seq;whole cell patch clamp',
 'SHARE-seq',
 'SMART-seq v4',
 'SNARE-seq2 (ATAC-seq)',
 'SNARE-seq2 (RNA-seq)',
 'SNARE-seq2;ATAC-seq',
 'SNARE-seq2;RNAseq',
 'Slide-seq',
 'mC-seq10',
 'mC-seq2',
 'mC-seq2;retrograde tracing',
 'mC-seq3',
 'mC-seq4',
 'mC-seq5',
 'mC-seq6',
 'mC-seq7',
 'mC-seq8',
 'mC-seq9',
 'transcrip

{"10x Chromium 3' v3 sequencing",
 'DNAseq',
 'MERFISH',
 'TRIO tracing',
 'VISor;light sheet microscopy',
 'fMOST',
 'neuron morphology reconstruction',
 'sci-RNA-seq3',
 'seqFISH',
 'whole cell patch clamp'}

{'DNAseq': ['tissue sample'],
 "10x Chromium 3' v3 sequencing": ['cell body',
  'cell nucleus',
  'cells',
  'library',
  'nuclei',
  'whole cell'],
 'VISor;light sheet microscopy': ['spinal cord'],
 'TRIO tracing': ['brain', 'brains', 'spinal cord', 'whole brain'],
 'whole cell patch clamp': ['neuron'],
 'MERFISH': ['brain slice', 'tissue sample', 'whole cell'],
 'fMOST': ['whole brain', 'whole cell'],
 'seqFISH': ['spinal cord slice'],
 'sci-RNA-seq3': ['specimen'],
 'neuron morphology reconstruction': ['reconstruction',
  'reconstructions',
  'whole cell']}